# Motor de filtrado de propiedades con SQL

Caso práctico: a partir de 4 tablas de una cartera de alojamientos, identificar con SQL qué propiedades cumplen simultáneamente los 12 requisitos de un cliente y responder 7 preguntas de negocio adicionales.

### Conexión a la base de datos

In [75]:
!pip install sqlalchemy
!pip install mysql-connector-python
!pip install openpyxl

In [76]:
from sqlalchemy import create_engine
import sqlalchemy
import mysql.connector as mysql
import pandas as pd

In [77]:
import os

# Parámetros de conexión (la contraseña se lee de una variable de entorno, no se guarda en el código)
user     = 'root'
password = os.environ.get('MYSQL_PASSWORD', '')
db = 'property_matching'
sqlServer = 'localhost'

# Creamos string de conexión
connStr = "mysql+mysqlconnector://" + user + ":" + password + "@" + sqlServer +  "/"  + db

# Conectamos con el servidor
conn = create_engine(connStr).connect()

In [78]:
from sqlalchemy import text

conn.execute(text("SHOW DATABASES;")).fetchall()

In [79]:
import os
from sqlalchemy import create_engine, text

user = "root"
password = os.environ.get("MYSQL_PASSWORD", "")
db = "property_matching"
sqlServer = "localhost"

connStr = f"mysql+mysqlconnector://{user}:{password}@{sqlServer}/{db}"
engine = create_engine(connStr)
conn = engine.connect()

conn.execute(text("SELECT DATABASE();")).fetchall()

In [80]:
conn.execute(text("SHOW TABLES;")).fetchall()

[('ALOJAMIENTO',), ('PRECIO',), ('PUNTUACION',), ('UBICACION',)]

### Carga de los archivos xlsx

In [81]:
!pip install openpyxl

In [82]:
df_aloj = pd.read_excel("data/ALOJAMIENTO.xlsx")
df_aloj.head()

,CODIGO_ALOJAMIENTO,NOMBRE,CANT_BANYOS,CANT_HAB,SALON,TERRAZA,AIREC_ACOND,PISCINA,SUPERFICIE
0,ALOJ001,Villa Sol del Mar,2,3,SI,SI,SI,SI,120
1,ALOJ002,Apartamento Las Palmeras,1,2,SI,NO,SI,NO,75
2,ALOJ003,Cabaña El Refugio,1,1,NO,SI,NO,NO,45
3,ALOJ004,Piso Centro Histórico,2,3,SI,SI,SI,NO,90
4,ALOJ005,Estudio Playa Blanca,1,1,NO,NO,SI,NO,40


In [83]:
df_aloj.to_sql("ALOJAMIENTO", con=engine, if_exists="replace", index=False)

20

In [84]:
conn.execute(text("SHOW TABLES;")).fetchall()

[('ALOJAMIENTO',), ('PRECIO',), ('PUNTUACION',), ('UBICACION',)]

In [85]:
with engine.begin() as connection:
    df_aloj.to_sql("ALOJAMIENTO", con=connection, if_exists="replace", index=False)

In [86]:
from sqlalchemy import inspect

inspector = inspect(engine)
inspector.get_table_names()

['ALOJAMIENTO', 'PRECIO', 'PUNTUACION', 'UBICACION']

In [87]:
from sqlalchemy import text

with engine.connect() as c:
    print(c.execute(text("SHOW TABLES;")).fetchall())
    print(c.execute(text("SELECT COUNT(*) FROM ALOJAMIENTO;")).fetchall())

[('ALOJAMIENTO',), ('PRECIO',), ('PUNTUACION',), ('UBICACION',)]
[(20,)]


In [88]:
conn.close()
conn = engine.connect()
conn.execute(text("SHOW TABLES;")).fetchall()

[('ALOJAMIENTO',), ('PRECIO',), ('PUNTUACION',), ('UBICACION',)]

In [89]:
df_ubi = pd.read_excel("data/UBICACION.xlsx")
df_ubi.to_sql("UBICACION", con=conn, if_exists="replace", index=False)

conn.execute(text("SHOW TABLES;")).fetchall()

[('ALOJAMIENTO',), ('PRECIO',), ('PUNTUACION',), ('UBICACION',)]

In [90]:
df_pre = pd.read_excel("data/PRECIO.xlsx")
df_pre.head()

,CODIGO_ALOJAMIENTO,PRECIO_ALQUILER,RESERVA,PORCENTAJE_RESERVA
0,ALOJ001,1800,SI,100
1,ALOJ002,780,SI,25
2,ALOJ003,800,NO,0
3,ALOJ004,1600,SI,25
4,ALOJ005,900,NO,0


In [91]:
df_pre.to_sql("PRECIO", con=conn, if_exists="replace", index=False)
conn.execute(text("SHOW TABLES;")).fetchall()

[('ALOJAMIENTO',), ('PRECIO',), ('PUNTUACION',), ('UBICACION',)]

In [92]:
df_pun = pd.read_excel("data/PUNTUACION.xlsx")
df_pun.head()

,CODIGO_ALOJAMIENTO,PUNTOS,AGENCIA,PUNTOS_AGENCIA
0,ALOJ001,5.0,GlobalHome,4.4
1,ALOJ002,4.2,AndesRenta,4.1
2,ALOJ003,4.1,AlquilaFacil,4.0
3,ALOJ004,4.6,CaribeRent,4.6
4,ALOJ005,4.4,RentaTop,3.8


In [93]:
df_pun.columns

Index(['CODIGO_ALOJAMIENTO', 'PUNTOS', 'AGENCIA', 'PUNTOS_AGENCIA'], dtype='object')

In [94]:
df_pun.to_sql("PUNTUACION", con=conn, if_exists="replace", index=False)
conn.execute(text("SHOW TABLES;")).fetchall()

[('ALOJAMIENTO',), ('PRECIO',), ('PUNTUACION',), ('UBICACION',)]

In [95]:
conn.execute(text("SELECT COUNT(*) FROM PUNTUACION;")).fetchall()

[(20,)]

### Una vez hecha la carga de los ficheros y crear las tablas, hacemos un chequeo de estructura:

In [96]:
print(conn.execute(text("DESCRIBE ALOJAMIENTO;")).fetchall())
print(conn.execute(text("DESCRIBE UBICACION;")).fetchall())
print(conn.execute(text("DESCRIBE PRECIO;")).fetchall())
print(conn.execute(text("DESCRIBE PUNTUACION;")).fetchall())

[('CODIGO_ALOJAMIENTO', b'text', 'YES', '', None, ''), ('NOMBRE', b'text', 'YES', '', None, ''), ('CANT_BANYOS', b'bigint', 'YES', '', None, ''), ('CANT_HAB', b'bigint', 'YES', '', None, ''), ('SALON', b'text', 'YES', '', None, ''), ('TERRAZA', b'text', 'YES', '', None, ''), ('AIREC_ACOND', b'text', 'YES', '', None, ''), ('PISCINA', b'text', 'YES', '', None, ''), ('SUPERFICIE', b'bigint', 'YES', '', None, '')]
[('CODIGO_ALOJAMIENTO', b'text', 'YES', '', None, ''), ('PAIS', b'text', 'YES', '', None, ''), ('CIUDAD', b'text', 'YES', '', None, ''), ('DIST_METRO', b'double', 'YES', '', None, ''), ('DIST_CENTRO', b'double', 'YES', '', None, '')]
[('CODIGO_ALOJAMIENTO', b'text', 'YES', '', None, ''), ('PRECIO_ALQUILER', b'bigint', 'YES', '', None, ''), ('RESERVA', b'text', 'YES', '', None, ''), ('PORCENTAJE_RESERVA', b'bigint', 'YES', '', None, '')]
[('CODIGO_ALOJAMIENTO', b'text', 'YES', '', None, ''), ('PUNTOS', b'double', 'YES', '', None, ''), ('AGENCIA', b'text', 'YES', '', None, ''), ('P

# Primera Parte:
Crear la query de cada uno de las exigencias del cliente. Se debe crear una query para cada punto. (7 queries en total)

#### Pasamos a la primera Query: 
"El cliente exige terraza y aire acondicionado, pero no quiere piscina"

In [97]:
conn.execute(text("""
SELECT CODIGO_ALOJAMIENTO, NOMBRE, TERRAZA, AIREC_ACOND, PISCINA
FROM ALOJAMIENTO
WHERE TERRAZA = 'SI'
  AND AIREC_ACOND = 'SI'
  AND PISCINA = 'NO';
""")).fetchall()

[('ALOJ004', 'Piso Centro Histórico', 'SI', 'SI', 'NO'),
 ('ALOJ010', 'Ático del Sol', 'SI', 'SI', 'NO'),
 ('ALOJ015', 'Dúplex Las Rocas', 'SI', 'SI', 'NO'),
 ('ALOJ018', 'Piso Parque Central', 'SI', 'SI', 'NO')]

### contamos con 4 alojamientos que cumplen las condiciones que exige el cliente

## Segunda Query:
“No se alojará en estudios que no tengan salón ni en propiedades con solo un baño.”

In [98]:
conn.execute(text("""
SELECT CODIGO_ALOJAMIENTO, NOMBRE, SALON, CANT_BANYOS
FROM ALOJAMIENTO
WHERE SALON = 'SI'
  AND CANT_BANYOS > 1;
""")).fetchall()

[('ALOJ001', 'Villa Sol del Mar', 'SI', 2),
 ('ALOJ004', 'Piso Centro Histórico', 'SI', 2),
 ('ALOJ006', 'Chalet Vista al Lago', 'SI', 3),
 ('ALOJ007', 'Dúplex Jardines del Sur', 'SI', 2),
 ('ALOJ009', 'Casa Rústica Montaña', 'SI', 2),
 ('ALOJ010', 'Ático del Sol', 'SI', 3),
 ('ALOJ012', 'Villa Palmeras', 'SI', 2),
 ('ALOJ014', 'Casa del Mar', 'SI', 3),
 ('ALOJ015', 'Dúplex Las Rocas', 'SI', 3),
 ('ALOJ017', 'Chalet Jardín Secreto', 'SI', 3),
 ('ALOJ018', 'Piso Parque Central', 'SI', 2),
 ('ALOJ019', 'Casa Río Claro', 'SI', 2)]

## Tercera QUERY:
“La propiedad debe tener como mínimo 80m2.”

In [99]:
conn.execute(text("""
SELECT CODIGO_ALOJAMIENTO, NOMBRE, SUPERFICIE
FROM ALOJAMIENTO
WHERE SUPERFICIE >= 80;
""")).fetchall()

[('ALOJ001', 'Villa Sol del Mar', 120),
 ('ALOJ004', 'Piso Centro Histórico', 90),
 ('ALOJ006', 'Chalet Vista al Lago', 150),
 ('ALOJ007', 'Dúplex Jardines del Sur', 130),
 ('ALOJ009', 'Casa Rústica Montaña', 85),
 ('ALOJ010', 'Ático del Sol', 88),
 ('ALOJ012', 'Villa Palmeras', 110),
 ('ALOJ013', 'Apartamento Moderno', 80),
 ('ALOJ014', 'Casa del Mar', 160),
 ('ALOJ015', 'Dúplex Las Rocas', 120),
 ('ALOJ017', 'Chalet Jardín Secreto', 180),
 ('ALOJ018', 'Piso Parque Central', 100),
 ('ALOJ019', 'Casa Río Claro', 95)]

## Cuarta QUERY:
“Como el cliente odia el ruido, no quiere alojarse en propiedades a menos de 1 km de una estación de metro ni a menos de 2 km del centro.”

In [100]:
conn.execute(text("""
SELECT a.CODIGO_ALOJAMIENTO, a.NOMBRE, u.DIST_METRO, u.DIST_CENTRO
FROM ALOJAMIENTO a
JOIN UBICACION u
    ON a.CODIGO_ALOJAMIENTO = u.CODIGO_ALOJAMIENTO
WHERE u.DIST_METRO >= 1
  AND u.DIST_CENTRO >= 2;
""")).fetchall()

[('ALOJ001', 'Villa Sol del Mar', 1.0, 2.0),
 ('ALOJ004', 'Piso Centro Histórico', 3.0, 4.0),
 ('ALOJ005', 'Estudio Playa Blanca', 4.0, 3.0),
 ('ALOJ006', 'Chalet Vista al Lago', 2.0, 2.0),
 ('ALOJ007', 'Dúplex Jardines del Sur', 1.0, 3.0),
 ('ALOJ008', 'Loft Urbano', 2.0, 3.0),
 ('ALOJ009', 'Casa Rústica Montaña', 2.0, 4.0),
 ('ALOJ010', 'Ático del Sol', 2.0, 5.0),
 ('ALOJ015', 'Dúplex Las Rocas', 2.0, 4.0),
 ('ALOJ016', 'Estudio Urbano', 2.0, 2.0),
 ('ALOJ017', 'Chalet Jardín Secreto', 1.0, 2.0),
 ('ALOJ018', 'Piso Parque Central', 2.0, 4.0),
 ('ALOJ019', 'Casa Río Claro', 2.0, 2.0),
 ('ALOJ020', 'Loft Estación Norte', 3.0, 2.0)]

## Quinta QUERY:
"El rango permitido para pagar es entre 1.500€ y 2.000€ la noche."

In [101]:
conn.execute(text("""
SELECT a.CODIGO_ALOJAMIENTO, a.NOMBRE, p.PRECIO_ALQUILER
FROM ALOJAMIENTO a
JOIN PRECIO p
    ON a.CODIGO_ALOJAMIENTO = p.CODIGO_ALOJAMIENTO
WHERE p.PRECIO_ALQUILER BETWEEN 1500 AND 2000;
""")).fetchall()

[('ALOJ001', 'Villa Sol del Mar', 1800),
 ('ALOJ004', 'Piso Centro Histórico', 1600),
 ('ALOJ008', 'Loft Urbano', 1500),
 ('ALOJ009', 'Casa Rústica Montaña', 1600),
 ('ALOJ010', 'Ático del Sol', 1750),
 ('ALOJ011', 'Cabaña Bosque Encantado', 1825),
 ('ALOJ012', 'Villa Palmeras', 1750),
 ('ALOJ013', 'Apartamento Moderno', 1652),
 ('ALOJ014', 'Casa del Mar', 1941),
 ('ALOJ015', 'Dúplex Las Rocas', 1800),
 ('ALOJ016', 'Estudio Urbano', 2000),
 ('ALOJ018', 'Piso Parque Central', 1900),
 ('ALOJ019', 'Casa Río Claro', 1570),
 ('ALOJ020', 'Loft Estación Norte', 1580)]

## Sexta QUERY:
"El cliente quiere que el porcentaje de reserva sea del 25%"

In [102]:
conn.execute(text("""
SELECT a.CODIGO_ALOJAMIENTO, a.NOMBRE, p.PORCENTAJE_RESERVA
FROM ALOJAMIENTO a
JOIN PRECIO p
    ON a.CODIGO_ALOJAMIENTO = p.CODIGO_ALOJAMIENTO
WHERE p.PORCENTAJE_RESERVA = 25;
""")).fetchall()

[('ALOJ002', 'Apartamento Las Palmeras', 25),
 ('ALOJ004', 'Piso Centro Histórico', 25),
 ('ALOJ008', 'Loft Urbano', 25),
 ('ALOJ010', 'Ático del Sol', 25),
 ('ALOJ015', 'Dúplex Las Rocas', 25),
 ('ALOJ018', 'Piso Parque Central', 25)]

## Septima QUERY:
"El cliente quiere que el alojamiento tenga más de 4,5 puntos de confianza y que la agencia que lo lleva tenga más de 4 puntos."

In [103]:
conn.execute(text("""
SELECT a.CODIGO_ALOJAMIENTO, a.NOMBRE, pu.PUNTOS, pu.AGENCIA, pu.PUNTOS_AGENCIA
FROM ALOJAMIENTO a
JOIN PUNTUACION pu
    ON a.CODIGO_ALOJAMIENTO = pu.CODIGO_ALOJAMIENTO
WHERE pu.PUNTOS > 4.5
  AND pu.PUNTOS_AGENCIA > 4;
""")).fetchall()

[('ALOJ001', 'Villa Sol del Mar', 5.0, 'GlobalHome', 4.4),
 ('ALOJ004', 'Piso Centro Histórico', 4.6, 'CaribeRent', 4.6),
 ('ALOJ007', 'Dúplex Jardines del Sur', 4.9, 'SevillaRooms', 4.5),
 ('ALOJ008', 'Loft Urbano', 4.8, 'ChileVive', 4.3),
 ('ALOJ009', 'Casa Rústica Montaña', 4.7, 'MexicoRent', 4.8),
 ('ALOJ010', 'Ático del Sol', 4.8, 'UYAlquila', 4.9),
 ('ALOJ014', 'Casa del Mar', 4.7, 'BogotaLodge', 4.5),
 ('ALOJ015', 'Dúplex Las Rocas', 4.9, 'EasyStay Valencia', 4.7),
 ('ALOJ017', 'Chalet Jardín Secreto', 4.9, 'CuscoRenta', 4.4),
 ('ALOJ018', 'Piso Parque Central', 4.7, 'MendozaRooms', 4.7),
 ('ALOJ019', 'Casa Río Claro', 4.9, 'GDLRent', 4.2),
 ('ALOJ020', 'Loft Estación Norte', 4.9, 'VivaStay', 4.9)]

# Segunda Parte:
Crear en una única query la combinación de los filtros anteriores para hallar las propiedades que finalmente cumplen con todas las exigencias del cliente.

In [104]:
conn.execute(text("""
SELECT a.CODIGO_ALOJAMIENTO,
       a.NOMBRE,
       a.SUPERFICIE,
       u.DIST_METRO,
       u.DIST_CENTRO,
       p.PRECIO_ALQUILER,
       p.PORCENTAJE_RESERVA,
       pu.PUNTOS,
       pu.AGENCIA,
       pu.PUNTOS_AGENCIA
FROM ALOJAMIENTO a
JOIN UBICACION u
    ON a.CODIGO_ALOJAMIENTO = u.CODIGO_ALOJAMIENTO
JOIN PRECIO p
    ON a.CODIGO_ALOJAMIENTO = p.CODIGO_ALOJAMIENTO
JOIN PUNTUACION pu
    ON a.CODIGO_ALOJAMIENTO = pu.CODIGO_ALOJAMIENTO
WHERE a.TERRAZA = 'SI'
  AND a.AIREC_ACOND = 'SI'
  AND a.PISCINA = 'NO'
  AND a.SALON = 'SI'
  AND a.CANT_BANYOS > 1
  AND a.SUPERFICIE >= 80
  AND u.DIST_METRO >= 1
  AND u.DIST_CENTRO >= 2
  AND p.PRECIO_ALQUILER BETWEEN 1500 AND 2000
  AND p.PORCENTAJE_RESERVA = 25
  AND pu.PUNTOS > 4.5
  AND pu.PUNTOS_AGENCIA > 4;
""")).fetchall()

[('ALOJ004', 'Piso Centro Histórico', 90, 3.0, 4.0, 1600, 25, 4.6, 'CaribeRent', 4.6),
 ('ALOJ010', 'Ático del Sol', 88, 2.0, 5.0, 1750, 25, 4.8, 'UYAlquila', 4.9),
 ('ALOJ015', 'Dúplex Las Rocas', 120, 2.0, 4.0, 1800, 25, 4.9, 'EasyStay Valencia', 4.7),
 ('ALOJ018', 'Piso Parque Central', 100, 2.0, 4.0, 1900, 25, 4.7, 'MendozaRooms', 4.7)]

#### Tras aplicar todos los filtros solicitados por el cliente mediante una query combinada con JOIN entre las tablas: 
'ALOJAMIENTO' 'UBICACION' 'PRECIO' y 'PUNTUACION'

Se obtuvieron 4 propiedades finalistas. Entre ellas, la opción más recomendable es ALOJ015 – Dúplex Las Rocas, ya que ofrece mayor superficie (120 m²) por la tanto posee una amplitud del espacio bastante considerable, una valoración del alojamiento muy alta (4.9) y una puntuación de agencia sólida (4.7) por lo que la convierte en la mejor opción, manteniéndose dentro del rango económico que solicita el cliente y cumpliendo todas las restricciones definidas por el mismo.

# Segunda Parte:

### Primera QUERY:
-	¿Cuál es la suma total de metros cuadrados de los alojamientos que tienen piscina?


In [105]:
conn.execute(text("""
SELECT SUM(SUPERFICIE) AS TOTAL_M2_PISCINA
FROM ALOJAMIENTO
WHERE PISCINA = 'SI';
""")).fetchall()

[(Decimal('850'),)]

### Segunda QUERY:
-	¿Cuál es el alojamiento con mayor superficie?

In [106]:
conn.execute(text("""
SELECT CODIGO_ALOJAMIENTO, NOMBRE, SUPERFICIE
FROM ALOJAMIENTO
WHERE SUPERFICIE = (
    SELECT MAX(SUPERFICIE)
    FROM ALOJAMIENTO
);
""")).fetchall()

[('ALOJ017', 'Chalet Jardín Secreto', 180)]

# Tercera QUERY:
-	¿Cuál es el alojamiento con menor superficie?

In [107]:
conn.execute(text("""
SELECT CODIGO_ALOJAMIENTO, NOMBRE, SUPERFICIE
FROM ALOJAMIENTO
WHERE SUPERFICIE = (
    SELECT MIN(SUPERFICIE)
    FROM ALOJAMIENTO
);
""")).fetchall()

[('ALOJ016', 'Estudio Urbano', 35)]

# Cuarta QUERY:
-	¿Cuántos alojamientos tienen salón y cuántos tienen terraza?

In [108]:
conn.execute(text("""
SELECT 
    CONCAT(COUNT(CASE WHEN SALON = 'SI' THEN 1 END), ' tienen salón') AS SALON,
    CONCAT(COUNT(CASE WHEN TERRAZA = 'SI' THEN 1 END), ' tienen terraza') AS TERRAZA
FROM ALOJAMIENTO;
""")).fetchall()

[('16 tienen salón', '14 tienen terraza')]

# Quinta QUERY:
-	¿Cuántos alojamientos hay por cada país según nuestra tabla de UBICACIÓN?

In [109]:
conn.execute(text("""
SELECT PAIS, COUNT(*) AS TOTAL_ALOJAMIENTOS
FROM UBICACION
GROUP BY PAIS
ORDER BY TOTAL_ALOJAMIENTOS DESC;
""")).fetchall()

[('España', 8), ('Portugal', 4), ('Francia', 4), ('Italia', 4)]

# Sexta QUERY:
-	¿Cuál es el promedio de precios de alquiler de todos los alojamientos según la tabla PRECIO?

In [110]:
conn.execute(text("""
SELECT CONCAT(
    'El promedio de precios de alquiler es: ',
    ROUND(AVG(PRECIO_ALQUILER), 2),
    ' €'
) AS PROMEDIO_PRECIO
FROM PRECIO;
""")).fetchall()

[('El promedio de precios de alquiler es: 1568.40 €',)]

# Septima QUERY:
-	¿Cuál o cuáles son las agencias que tienen la mayor puntuación de agencia?

In [111]:
conn.execute(text("""
SELECT AGENCIA, PUNTOS_AGENCIA
FROM PUNTUACION
WHERE PUNTOS_AGENCIA = (
    SELECT MAX(PUNTOS_AGENCIA)
    FROM PUNTUACION
);
""")).fetchall()

[('TerraHouse', 4.9),
 ('UYAlquila', 4.9),
 ('MalagaTurismo', 4.9),
 ('VivaStay', 4.9)]